In [14]:
import asyncio
from mcp import ClientSession
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client


In [ ]:
server_params = StdioServerParameters(
    command="uvx",
    args=[
        "mcp-server-fetch",
        "--user-agent=Mozilla/5.0"
        # The README explicitly says custom user agents are supported.in the context of mcp where i took it!
    ]
)

In [48]:
async def discover_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools=await session.list_tools()
            print("Available Tools:")

            for tool in tools.tools:
                print(tool.name)

In [49]:
await discover_tools()

Available Tools:
fetch


In [50]:
async def fetch_tool(url):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "fetch",
                {
                    "url": url,
                    "max_length": 5000,
                    "raw": True
                }
            )

            if result.isError:
                print("MCP ERROR:")
                print(result)
                return None

            return result.content[0].text

In [51]:
async def react_agent(url):
    print("thought:")
    print("need Information from website")

    print("\nAction:")
    print("fetch")

    print("\nAction Input:")
    print(url)

    observation= await fetch_tool(url)
    print(observation[:300])
    return observation[:500]

    # return observation[:500]

In [52]:
answer= await react_agent("https://openai.com" )
print(answer)

thought:
need Information from website

Action:
fetch

Action Input:
https://openai.com
MCP ERROR:
meta=None content=[TextContent(type='text', text='Failed to fetch https://openai.com/ - status code 403', annotations=None, meta=None)] structuredContent=None isError=True


TypeError: 'NoneType' object is not subscriptable

In [54]:
ans=await react_agent("https://www.python.org")
print(ans)

thought:
need Information from website

Action:
fetch

Action Input:
https://www.python.org


Content type text/html; charset=utf-8 cannot be simplified to markdown, but here is the raw content:
Contents of https://www.python.org/:
<!doctype html>
<html class="no-js" lang="en" dir="ltr">

<head>
    <script defer
            file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip"
  
Content type text/html; charset=utf-8 cannot be simplified to markdown, but here is the raw content:
Contents of https://www.python.org/:
<!doctype html>
<html class="no-js" lang="en" dir="ltr">

<head>
    <script defer
            file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip"
            data-domain="python.org"
            src="https://analytics.python.org/js/script.file-downloads.outbound-links.js"></script>

    <meta charset="utf-8">

    <link rel="prefetch" href="//aja


using llm to simplify the matter generated from the website


In [55]:
import os
from dotenv import load_dotenv

load_dotenv("myenv.env")

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [57]:


prompt = f"""
You are a web analyst.

Summarize the content:

{ans}
"""

response = llm.invoke(prompt)

print(response.content)

Certainly! Here's a summarized version of the HTML content for the Python website homepage:

---

**HTML Document for Python.org:**

- **DOCTYPE Declaration:** `<!doctype html>`
- **HTML Tag:** 
  - `class="no-js"`
  - `lang="en"`
  - `dir="ltr"` (left-to-right text direction)
  
**Head Section:**

- **Script Tag:** 
  - `defer` attribute
  - Specifies acceptable file types for downloads (compressed files, executables, etc.)
  - Domain set to `python.org`
  - Source script from analytics endpoint
  
- **Meta Tag:**
  - Specifies character set as `utf-8`
  
- **Link Tag:**
  - `rel="prefetch"`
  - Specifies a resource to be fetched in advance (`href` attribute is incomplete in the provided snippet)

---

This summary captures the primary structure and elements found in the provided HTML snippet from the Python website.
